The idea is to first plan a route && create the speed profile according to the lane then:
- Check for stop signs that apply
- If they apply -> truncate the path at the first stop sign
- Set a speed profile element at the final waypoint with speed = 0

The planner will need to keep track of it's state. From P&C Design:
- There are 3 states: Driving, Slowing, Stopped
- Only the following transitions are allowed:
    - Driving --> Slowing
    - Slowing --> Stopped
    - Stopped --> Driving

Important changes:
- Speed profile and planned path are consolidated into a single data structure `route`.
    - Route is a list of waypoints. Each waypoint has a position and target speed.

This *does* need changes!!
Currently Planning knows where the car is in absolute terms. This is not how our planning module actually works and a transformation is needed there before merging this into planning/moving further.

This probably isn't that big of a deal but it does need to be done no matter what.

In [4]:
from types import SimpleNamespace

"""Simple 1D simulator"""
class Sim:
    def __init__(self):
        self.x = 0 # start at x = 0
        self.speed = 0 # start with v = 0
        self.signs = [10, 20, 30] # stop sign every 10 meters

        # Constants
        self.MIN_SPEED = 0.0001
    
    def update(self, dt: float, acc: float=0) -> SimpleNamespace:
        # update speed
        self.speed += acc * dt
        if self.speed < self.MIN_SPEED:  # snap to 0 if close enough
            self.speed = 0
        
        # update position
        self.x += self.speed * dt

        return self.get_frame()

    def get_frame(self) -> SimpleNamespace:
        """Outputs a frame in a coordinate space centered around the car."""
        frame = SimpleNamespace()
        frame.speed = self.speed 
        frame.signs = [sign - self.x for sign in self.signs]
        frame.odometer = self.x # total distance covered

        return frame


In [9]:
import time
from time import sleep

class Control:
    def __init__(self):
        # constants
        self.MAX_ACCELERATION = 3 # ms^-2
        self.EPSILON = 0.001

    # Waypoints are (distance, speed)
    def update(self, current_speed: float, next_waypoint: tuple[float, float]) -> float:
        def determine_required_acceleration(v0: float, vf: float, d: float):
            if d == 0: # this means just set the speed ASAP
                # if we're already there no acceleration
                if abs(vf - v0) < self.EPSILON:
                    return 0
                
                # otherwise, return self.MAX_ACCELERATION in the direction we need to go
                if vf > v0:
                    return self.MAX_ACCELERATION
                else:
                    return -self.MAX_ACCELERATION

            # if there is a distance to the upcoming route node
            # use that to calculate the required acceleration
            return (vf**2 - v0**2) / (2 * d)

        acc = determine_required_acceleration(
            current_speed, next_waypoint[1], # v0, vf
            next_waypoint[0] - 0 # ds
        )

        return acc

class Planner:
    def __init__(self):
        # State
        self._state = self.driving
        
        # Constants
        self.STOPPING_TRANSITION_DISTANCE = 3 # m
        self.DEFAULT_SPEED = 2 # ms^-1
        self.MAX_BRAKING = 2 # ms^-2
        self.MIN_STOP_DURATION = 3 # s
        self.DRIVE_THROUGH_DISTANCE = 1 # m

    """
    States are represented by functions in this prototype.
    Transitions are relatively logical:
    - when 'DRIVING' stay driving unless we see a stop sign
    - when 'STOPPING' stay STOPPING until v = 0
    - when 'STOPPED' stay 'STOPPED' until time has elapsed
    - when 'DRIVING_THROUGH' stay driving until some distance has been covered (to get away from previous stop sign)

    And behaviours are also simple:
    - when 'DRIVING' or 'DRIVING_THROUGH', drive forward at target speed
    - when 'STOPPING', lerp (or something) until v = 0
    - when 'STOPPED', v = 0

    Each state returns the next state.
    """
    def driving(self, frame): # frame is a collection of data
        def sign_is_close(signs: list[float]) -> bool:
            for sign_x in signs:
                # skip signs behind us
                if sign_x < 0:
                    continue 

                # if the sign is close by
                if sign_x < self.STOPPING_TRANSITION_DISTANCE:
                    return True 
                
            return False 
    
        # Are we close to a sign?
        if sign_is_close(frame.signs):
            # transition to stopping right away
            return self.stopping(frame)
        else:
            # otherwise just create a regular route
            route = [(i, self.DEFAULT_SPEED) for i in range(1, 4)] # 1 waypoint every meter at the default speed
            return route, self.driving
    
    def driving_through(self, frame):
        if frame.odometer - self.drive_through_start > self.DRIVE_THROUGH_DISTANCE:
            return self.driving(frame)

        route = [(i, self.DEFAULT_SPEED) for i in range(1, 4)]
        return route, self.driving_through
    
    def stopping(self, frame):
        # If we're stopped
        if frame.speed <= 0:
            # start the timer and transition immediately
            self.stop_time = time.perf_counter()
            return self.stopped(frame)
        
        # Otherwise,
        # 0. Get the total stopping distance
        def get_next_sign(signs: list[float]) -> float:
            # filter by those ahead
            signs_ahead = [sign for sign in signs if sign > 0]
            
            # return the smallest
            if len(signs_ahead) == 0:
                return -1
            else:
                return min(signs_ahead)

        sign_x = get_next_sign(frame.signs)
        if sign_x == -1:
            raise RuntimeError("Stopping with no upcoming sign?")
            # maybe transition to driving
        # d_stop = sign_x - frame.x
            
        # 1. Calculate braking distance
        # d_brake = frame.speed ** 2 / (2 * self.MAX_BRAKING)
        
        # 2. Calculate brake start location
        # start_brake = d_stop - d_brake 

        # 3. Construct Route [
        # ( start_brake (m), v0 (m/s) ),
        # ( d_stop (m), 0 (m/s) )
        # ]
        route = [(sign_x, 0)] # should be a little more complicated right? maybe to smooth in the future?

        # stay in stopping
        return route, self.stopping

    def stopped(self, frame):
        # if enough time has elapsed
        now = time.perf_counter()
        stop_duration = now - self.stop_time
        if stop_duration > self.MIN_STOP_DURATION:
            # transition immediately to driving_through
            self.drive_through_start = frame.odometer
            return self.driving_through(frame)
        
        # otherwise stay stopped
        route = [(0, 0)]
        return route, self.stopped 

    def update(self, frame) -> list[float, float]:
        route, self._state = self._state(frame)
        return route


class PNC:
    def __init__(self):
        self.control = Control()
        self.planning = Planner()
        self.sim = Sim()

    def run(self, DT):
        acc = 0 # loop carryover mem
        while self.sim.x < max(self.sim.signs):
            # do the actual loop
            frame = self.sim.update(DT, acc)
            route = self.planning.update(frame)
            acc = self.control.update(self.sim.speed, route[0])
            
            # to see if it's actually working
            print(f'[{int(time.time() % 12309)}] Pos: {self.sim.x:.2f}m, V: {self.sim.speed:.2f}m/s, State: {self.planning._state.__name__}')
            # delay before next iteration
            sleep(DT)



In [10]:
pnc = PNC()
pnc.run(0.2)

[9163] Pos: 0.00m, V: 0.00m/s, State: driving
[9163] Pos: 0.08m, V: 0.40m/s, State: driving
[9164] Pos: 0.24m, V: 0.78m/s, State: driving
[9164] Pos: 0.46m, V: 1.12m/s, State: driving
[9164] Pos: 0.74m, V: 1.40m/s, State: driving
[9164] Pos: 1.06m, V: 1.60m/s, State: driving
[9164] Pos: 1.41m, V: 1.75m/s, State: driving
[9165] Pos: 1.78m, V: 1.84m/s, State: driving
[9165] Pos: 2.16m, V: 1.90m/s, State: driving
[9165] Pos: 2.55m, V: 1.94m/s, State: driving
[9165] Pos: 2.94m, V: 1.96m/s, State: driving
[9165] Pos: 3.33m, V: 1.98m/s, State: driving
[9166] Pos: 3.73m, V: 1.99m/s, State: driving
[9166] Pos: 4.13m, V: 1.99m/s, State: driving
[9166] Pos: 4.53m, V: 2.00m/s, State: driving
[9166] Pos: 4.93m, V: 2.00m/s, State: driving
[9166] Pos: 5.33m, V: 2.00m/s, State: driving
[9167] Pos: 5.73m, V: 2.00m/s, State: driving
[9167] Pos: 6.13m, V: 2.00m/s, State: driving
[9167] Pos: 6.53m, V: 2.00m/s, State: driving
[9167] Pos: 6.93m, V: 2.00m/s, State: driving
[9167] Pos: 7.33m, V: 2.00m/s, Sta